# Phase 11 — Ensemble 3x DistilBERT (different seeds)

Συνδυασμός 3 DistilBERT μοντέλων εκπαιδευμένων σε train+valid με διαφορετικά seeds.

**Μοντέλα:**
- DistilBERT seed original → `bert_fulldata_test_*_probs.npy`
- DistilBERT seed 123     → `bert_fulldata_test_*_probs123.npy`
- DistilBERT seed 456     → `bert_fulldata_test_*_probs456.npy`

**Μέθοδος:** Simple average των probabilities (κάθε μοντέλο βάρος 1/3)

In [2]:
import numpy as np
import pandas as pd

test = pd.read_csv('test.csv')

# Φόρτωση test probabilities από τα 3 μοντέλα
hazard_probs_orig  = np.load('npy/bert_fulldata_test_hazard_probs.npy')
hazard_probs_s123  = np.load('npy/bert_fulldata_test_hazard_probs123.npy')
hazard_probs_s456  = np.load('npy/bert_fulldata_test_hazard_probs456.npy')

product_probs_orig = np.load('npy/bert_fulldata_test_product_probs.npy')
product_probs_s123 = np.load('npy/bert_fulldata_test_product_probs123.npy')
product_probs_s456 = np.load('npy/bert_fulldata_test_product_probs456.npy')

print(f'Hazard probs shape:  {hazard_probs_orig.shape}')
print(f'Product probs shape: {product_probs_orig.shape}')

Hazard probs shape:  (997, 10)
Product probs shape: (997, 22)


In [3]:
# Χρειαζόμαστε τα label classes για να κάνουμε inverse_transform
import pandas as pd
from sklearn.preprocessing import LabelEncoder

train = pd.read_csv('train.csv')
valid = pd.read_csv('valid.csv')
train_full = pd.concat([train, valid], ignore_index=True)

le_hazard  = LabelEncoder()
le_product = LabelEncoder()
le_hazard.fit(train_full['hazard-category'])
le_product.fit(train_full['product-category'])

print(f'Hazard classes:  {list(le_hazard.classes_)}')
print(f'Product classes: {len(le_product.classes_)}')

Hazard classes:  ['allergens', 'biological', 'chemical', 'food additives and flavourings', 'foreign bodies', 'fraud', 'migration', 'organoleptic aspects', 'other hazard', 'packaging defect']
Product classes: 22


In [ ]:
# Simple average των 3 μοντέλων
hazard_avg  = (hazard_probs_orig  + hazard_probs_s123  + hazard_probs_s456)  / 3
product_avg = (product_probs_orig + product_probs_s123 + product_probs_s456) / 3

# Predictions
test_hazard  = le_hazard.inverse_transform(hazard_avg.argmax(axis=1))
test_product = le_product.inverse_transform(product_avg.argmax(axis=1))

# Submission
submission = pd.DataFrame({
    'id': test['id'],
    'hazard-category': test_hazard,
    'product-category': test_product
})
submission.to_csv('submission_3bert_ensemble.csv', index=False)

print('Αποθηκεύτηκε: submission_3bert_ensemble.csv')
print(f'Predictions: {len(submission)}')
print('\n--- Πρώτες 5 ---')
print(submission.head())
print('\n=== ΣΥΓΚΡΙΣΗ ===')
print(f'Best single model (20 epochs): 0.76064')

Αποθηκεύτηκε: submission_3bert_ensemble.csv
Predictions: 997

--- Πρώτες 5 ---
   id hazard-category              product-category
0   0      biological  meat, egg and dairy products
1   1      biological  meat, egg and dairy products
2   2      biological    prepared dishes and snacks
3   3      biological  meat, egg and dairy products
4   4  foreign bodies             ices and desserts

=== ΣΥΓΚΡΙΣΗ ===
Best single model (20 epochs): 0.76064
→ Κάνε submit για να δεις το νέο score!
